# Diffusion Reconstruction

In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BATCH_SIZE = 128
IMG_SIZE = 32
TIMESTEPS = 200

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_ds = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_ds = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
def sinusoidal_embedding(t, dim):
    half = dim // 2
    emb_scale = math.log(10000) / (half - 1)
    emb = torch.exp(torch.arange(half, device=t.device) * -emb_scale)
    emb = t[:, None].float() * emb[None, :]
    emb = torch.cat([emb.sin(), emb.cos()], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)
        h = h + self.t_proj(t_emb)[:, :, None, None]
        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)
        return h + self.skip(x)

class SmallUNet(nn.Module):
    def __init__(self, in_ch=3, base=64, t_dim=128):
        super().__init__()
        self.t_mlp = nn.Sequential(
            nn.Linear(t_dim, t_dim * 2),
            nn.SiLU(),
            nn.Linear(t_dim * 2, t_dim),
        )
        self.in_conv = nn.Conv2d(in_ch, base, 3, padding=1)
        self.down1 = ResBlock(base, base, t_dim)
        self.down2 = ResBlock(base, base * 2, t_dim)
        self.pool = nn.AvgPool2d(2)
        self.mid = ResBlock(base * 2, base * 2, t_dim)
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.up1 = ResBlock(base * 2 + base * 2, base, t_dim)
        self.up2 = ResBlock(base + base, base, t_dim)
        self.out = nn.Conv2d(base, in_ch, 3, padding=1)
        self.t_dim = t_dim

    def forward(self, x, t):
        t_emb = self.t_mlp(sinusoidal_embedding(t, self.t_dim))
        x0 = self.in_conv(x)
        d1 = self.down1(x0, t_emb)
        d2 = self.down2(self.pool(d1), t_emb)
        m = self.mid(d2, t_emb)
        u1 = self.up(m)
        u1 = torch.cat([u1, d2], dim=1)
        u1 = self.up1(u1, t_emb)
        u2 = torch.cat([u1, d1], dim=1)
        u2 = self.up2(u2, t_emb)
        return self.out(u2)

class GaussianDiffusion:
    def __init__(self, timesteps=200, beta_start=1e-4, beta_end=0.02, device='cpu'):
        self.timesteps = timesteps
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = torch.cat([torch.ones(1, device=device), self.alphas_cumprod[:-1]])
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.posterior_var = self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)

    def _extract(self, a, t, x_shape):
        return a.gather(0, t).view(-1, 1, 1, 1).expand(x_shape)

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        return self._extract(self.sqrt_alphas_cumprod, t, x0.shape) * x0 + self._extract(self.sqrt_one_minus_alphas_cumprod, t, x0.shape) * noise

    def predict_x0(self, x_t, t, pred_noise):
        x0 = (x_t - self._extract(self.sqrt_one_minus_alphas_cumprod, t, x_t.shape) * pred_noise) / self._extract(self.sqrt_alphas_cumprod, t, x_t.shape)
        return torch.clamp(x0, -1.0, 1.0)

    @torch.no_grad()
    def p_sample(self, model, x_t, t):
        pred_noise = model(x_t, t)
        beta_t = self._extract(self.betas, t, x_t.shape)
        sqrt_one_minus = self._extract(self.sqrt_one_minus_alphas_cumprod, t, x_t.shape)
        sqrt_recip_alpha = self._extract(self.sqrt_recip_alphas, t, x_t.shape)
        mean = sqrt_recip_alpha * (x_t - beta_t / sqrt_one_minus * pred_noise)
        var = self._extract(self.posterior_var, t, x_t.shape)
        nonzero = (t != 0).float().view(-1, 1, 1, 1)
        return mean + nonzero * torch.sqrt(torch.clamp(var, min=1e-20)) * torch.randn_like(x_t)

    @torch.no_grad()
    def sample(self, model, shape, device):
        x = torch.randn(shape, device=device)
        for step in reversed(range(self.timesteps)):
            t = torch.full((shape[0],), step, device=device, dtype=torch.long)
            x = self.p_sample(model, x, t)
        return x


In [ ]:
@torch.no_grad()
def update_ema(ema_model, model, decay=0.999):
    for ema_p, p in zip(ema_model.parameters(), model.parameters()):
        ema_p.data.mul_(decay).add_(p.data, alpha=1.0 - decay)

def train_model(epochs=20, lr=2e-4):
    model = SmallUNet().to(device)
    ema_model = SmallUNet().to(device)
    ema_model.load_state_dict(model.state_dict())
    ema_model.eval()

    diffusion = GaussianDiffusion(timesteps=TIMESTEPS, device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    model.train()
    for epoch in range(epochs):
        running = 0.0
        for x0, _ in train_loader:
            x0 = x0.to(device, non_blocking=True)
            t = torch.randint(0, diffusion.timesteps, (x0.size(0),), device=device)
            noise = torch.randn_like(x0)
            x_t = diffusion.q_sample(x0, t, noise)
            pred_noise = model(x_t, t)
            loss = F.mse_loss(pred_noise, noise)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            update_ema(ema_model, model)
            running += loss.item()

        print(f"Epoch {epoch + 1:02d}/{epochs} | loss={running / len(train_loader):.4f}")

    return model, ema_model, diffusion

model, ema_model, diffusion = train_model(epochs=20, lr=2e-4)


In [ ]:
def denorm(x):
    return (x.clamp(-1, 1) + 1.0) / 2.0

@torch.no_grad()
def show_reconstruction(ema_model, diffusion, loader, t_level=120, n=8):
    x0, _ = next(iter(loader))
    x0 = x0[:n].to(device)
    t = torch.full((n,), t_level, device=device, dtype=torch.long)
    noise = torch.randn_like(x0)
    x_t = diffusion.q_sample(x0, t, noise)
    pred_noise = ema_model(x_t, t)
    x0_hat = diffusion.predict_x0(x_t, t, pred_noise)

    fig, axes = plt.subplots(3, n, figsize=(2*n, 6))
    for i in range(n):
        axes[0, i].imshow(denorm(x0[i]).permute(1, 2, 0).cpu())
        axes[1, i].imshow(denorm(x_t[i]).permute(1, 2, 0).cpu())
        axes[2, i].imshow(denorm(x0_hat[i]).permute(1, 2, 0).cpu())
        axes[0, i].axis('off'); axes[1, i].axis('off'); axes[2, i].axis('off')
    axes[0, 0].set_title('Original')
    axes[1, 0].set_title('Noisy x_t')
    axes[2, 0].set_title('Reconstruction')
    plt.tight_layout(); plt.show()

@torch.no_grad()
def show_generated(ema_model, diffusion, n=8):
    samples = diffusion.sample(ema_model, shape=(n, 3, IMG_SIZE, IMG_SIZE), device=device)
    fig, axes = plt.subplots(1, n, figsize=(2*n, 2.4))
    for i in range(n):
        axes[i].imshow(denorm(samples[i]).permute(1, 2, 0).cpu())
        axes[i].axis('off')
    plt.tight_layout(); plt.show()

show_reconstruction(ema_model, diffusion, test_loader, t_level=120, n=8)
show_generated(ema_model, diffusion, n=8)
